# Growing Neural Cellular Automata — MICCAI 2026 prototype

This notebook trains a tiny neural cellular automaton (NCA) to grow the MICCAI 2026 France logo from a single living cell. It is intentionally compact: one target image, one model, one training loop, and visual feedback throughout.

**Colab:** choose **Runtime → Change runtime type → GPU**, then run all cells. The default settings are designed for a quick experiment; increase `training_steps` for a cleaner result.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoKalk/colab_test_miccai2026/blob/main/growing_nca_colab.ipynb)

## 1. Setup

Colab already includes PyTorch, Pillow, NumPy, and Matplotlib, so there is nothing to install.

In [ ]:
import io
import random
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import animation
from PIL import Image
from IPython.display import HTML, clear_output, display

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}. A Colab GPU is recommended.")

## 2. Experiment controls

These fields become interactive controls in Google Colab. The default 32×32 canvas is intentionally small so the experiment trains quickly. For a more polished result, try `training_steps = 1200`.

In [ ]:
image_size = 32  #@param {type:"slider", min:24, max:64, step:8}
training_steps = 500  #@param {type:"slider", min:100, max:1500, step:100}
batch_size = 4  #@param {type:"slider", min:1, max:8, step:1}
learning_rate = 0.002  #@param {type:"number"}
min_growth_steps = 24  #@param {type:"slider", min:16, max:48, step:4}
max_growth_steps = 40  #@param {type:"slider", min:24, max:64, step:4}
preview_every = 50  #@param {type:"slider", min:10, max:200, step:10}

assert min_growth_steps <= max_growth_steps
print(f"Target: {image_size}×{image_size} | training updates: {training_steps}")

## 3. Prepare the target image

The logo is fitted into a square transparent canvas while preserving its aspect ratio. To use another target later, replace the PNG in `assets/` or change `TARGET_URL`.

In [ ]:
TARGET_URL = (
    "https://raw.githubusercontent.com/JoKalk/"
    "colab_test_miccai2026/main/assets/miccai2026-logo-france.png"
)
local_target = Path("assets/miccai2026-logo-france.png")

if local_target.exists():
    source_image = Image.open(local_target).convert("RGBA")
else:
    with urllib.request.urlopen(TARGET_URL) as response:
        source_image = Image.open(io.BytesIO(response.read())).convert("RGBA")

def prepare_target(image, size, margin=2):
    image = image.copy()
    image.thumbnail((size - 2 * margin, size - 2 * margin), Image.Resampling.LANCZOS)
    canvas = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    offset = ((size - image.width) // 2, (size - image.height) // 2)
    canvas.paste(image, offset, image)
    array = np.asarray(canvas).astype(np.float32) / 255.0
    return torch.from_numpy(array).to(device), canvas

target, target_preview = prepare_target(source_image, image_size)
display(target_preview.resize((320, 320), Image.Resampling.NEAREST))

## 4. Define the growing NCA

Each cell stores RGBA plus hidden channels. It sees itself and local horizontal/vertical gradients, then a small neural network decides how to update the cell. Cells survive only near visible alpha, which creates the growing behavior.

In [ ]:
CHANNELS = 12
HIDDEN_SIZE = 32
FIRE_RATE = 0.5

class GrowingNCA(nn.Module):
    def __init__(self, channels=CHANNELS, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.channels = channels
        self.update_net = nn.Sequential(
            nn.Conv2d(channels * 3, hidden_size, 1),
            nn.ReLU(),
            nn.Conv2d(hidden_size, channels, 1, bias=False),
        )
        nn.init.zeros_(self.update_net[-1].weight)

        identity = torch.tensor([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=torch.float32)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32) / 8
        sobel_y = sobel_x.T
        kernels = torch.stack([identity, sobel_x, sobel_y]).unsqueeze(1)
        self.register_buffer("perception_kernels", kernels.repeat(channels, 1, 1, 1))

    def alive_mask(self, state):
        alpha = state[:, 3:4]
        return F.max_pool2d(alpha, kernel_size=3, stride=1, padding=1) > 0.1

    def perceive(self, state):
        return F.conv2d(state, self.perception_kernels, padding=1, groups=self.channels)

    def forward(self, state, fire_rate=FIRE_RATE):
        pre_life = self.alive_mask(state)
        update = self.update_net(self.perceive(state))
        stochastic_mask = (torch.rand_like(state[:, :1]) <= fire_rate).float()
        state = state + update * stochastic_mask
        post_life = self.alive_mask(state)
        return state * (pre_life & post_life).float()

def make_seed(batch=1):
    state = torch.zeros(batch, CHANNELS, image_size, image_size, device=device)
    state[:, 3, image_size // 2, image_size // 2] = 1.0
    return state

def to_rgb(state):
    rgba = state[0, :4].detach().permute(1, 2, 0).cpu().clamp(0, 1).numpy()
    return rgba[..., :3] * rgba[..., 3:] + (1.0 - rgba[..., 3:])

model = GrowingNCA().to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Train with live visual feedback

Every `preview_every` updates, this cell refreshes the loss curve and shows one deterministic growth sample. The preview is the live training output—not a precomputed result.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
target_batch = target.permute(2, 0, 1).unsqueeze(0)
loss_history = []

def grow(state, steps, fire_rate=FIRE_RATE):
    for _ in range(steps):
        state = model(state, fire_rate=fire_rate)
    return state

def show_training(step):
    model.eval()
    with torch.no_grad():
        preview = grow(make_seed(), max_growth_steps, fire_rate=0.5)
    model.train()

    clear_output(wait=True)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    axes[0].imshow(target_preview)
    axes[0].set_title("Target")
    axes[1].imshow(to_rgb(preview))
    axes[1].set_title(f"NCA after {max_growth_steps} steps")
    axes[2].plot(loss_history)
    axes[2].set_yscale("log")
    axes[2].set_title(f"Training loss — update {step}")
    axes[2].set_xlabel("optimizer update")
    for axis in axes[:2]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

model.train()
for step in range(1, training_steps + 1):
    state = make_seed(batch_size)
    rollout_steps = random.randint(min_growth_steps, max_growth_steps)
    state = grow(state, rollout_steps)

    loss = F.mse_loss(state[:, :4], target_batch.expand(batch_size, -1, -1, -1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    loss_history.append(loss.item())

    if step == 1 or step % preview_every == 0 or step == training_steps:
        show_training(step)

print(f"Finished. Final loss: {loss_history[-1]:.6f}")

## 6. Inspect growth after training

The first view shows snapshots across time. The second view is an animation of the same single seed developing into the target.

In [ ]:
model.eval()
snapshot_steps = [0, 6, 12, 18, 24, 32, 40, 48]
snapshots = []
state = make_seed()

with torch.no_grad():
    for step in range(max(snapshot_steps) + 1):
        if step in snapshot_steps:
            snapshots.append(to_rgb(state))
        state = model(state, fire_rate=0.5)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, step in zip(axes.flat, snapshots, snapshot_steps):
    axis.imshow(image)
    axis.set_title(f"step {step}")
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
animation_steps = 56
frames = []
state = make_seed()
with torch.no_grad():
    for step in range(animation_steps + 1):
        if step % 2 == 0:
            frames.append(to_rgb(state))
        state = model(state, fire_rate=0.5)

fig, axis = plt.subplots(figsize=(5, 5))
axis.axis("off")
artist = axis.imshow(frames[0], interpolation="nearest")

def animate(frame):
    artist.set_data(frame)
    return (artist,)

growth_animation = animation.FuncAnimation(
    fig, animate, frames=frames, interval=100, blit=True
)
plt.close(fig)
display(HTML(growth_animation.to_jshtml()))

## What to try next

- Increase `training_steps` to improve fidelity.
- Change the target PNG to a medical structure or icon.
- Add a state pool and damage examples to teach regeneration, as in the full Growing NCA experiment.
- Save model weights once the target and settings are stable.